# Model Viewer App - One Click Install

Run the cell below to deploy the **Model Viewer** app to your workspace.  
It will create the app, deploy it, and print the shareable URL.

In [ ]:
APP_NAME = "model-viewer-app"

import time, requests

ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
host, token = ctx.apiUrl().get(), ctx.apiToken().get()
H = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}

nb = ctx.notebookPath().get()
ws_logical = "/".join(nb.split("/")[:-1]) + "/model-viewer-app"
src_fs = ws_logical if ws_logical.startswith("/Workspace/") else "/Workspace" + ws_logical

r = requests.get(f"{host}/api/2.0/workspace/get-status", headers=H, params={"path": f"{ws_logical}/app.yaml"})
assert r.status_code == 200, (
    f"App source not found at {ws_logical}/app.yaml.\n"
    f"This installer expects the sibling 'model-viewer-app/' folder next to it.\n"
    f"Sync the full 'viewer/' folder from the repo into your workspace and re-run.\n"
    f"workspace get-status response: {r.status_code} {r.text}"
)
print(f"[1/4] Source verified: {ws_logical}")
print(f"       Apps API source_code_path: {src_fs}")

def _wait_compute_active(timeout_s=600):
    deadline = time.time() + timeout_s
    last = ""
    while time.time() < deadline:
        info = requests.get(f"{host}/api/2.0/apps/{APP_NAME}", headers=H).json()
        cs = info.get("compute_status", {}).get("state", "")
        if cs != last:
            print(f"       Compute: {cs}")
            last = cs
        if cs == "ACTIVE":
            return info
        if cs in ("ERROR", "STOPPED"):
            raise Exception(f"Compute entered {cs} while waiting; aborting deploy.")
        time.sleep(10)
    raise Exception(f"Compute did not reach ACTIVE within {timeout_s}s (last state={last!r}).")

r = requests.get(f"{host}/api/2.0/apps/{APP_NAME}", headers=H)
if r.status_code == 200:
    print(f"[2/4] App '{APP_NAME}' exists, ensuring compute is running...")
    cs = r.json().get("compute_status", {}).get("state", "")
    if cs != "ACTIVE":
        if cs in ("STOPPED", "ERROR", ""):
            sr = requests.post(f"{host}/api/2.0/apps/{APP_NAME}/start", headers=H, json={})
            assert sr.status_code in (200, 201, 202), f"Failed to start app compute: {sr.status_code} {sr.text}"
        _wait_compute_active()
elif r.status_code == 404:
    print(f"[2/4] Creating app '{APP_NAME}'...")
    cr = requests.post(f"{host}/api/2.0/apps", headers=H, json={"name": APP_NAME, "description": "Data Model Viewer - interactive graph visualization"})
    assert cr.status_code in (200, 201), f"Failed to create app: {cr.status_code} {cr.text}"
    print(f"       App created. Waiting for compute to become ACTIVE...")
    _wait_compute_active()
else:
    raise Exception(f"Unexpected response checking app: {r.status_code} {r.text}")

print(f"[3/4] Deploying...")
r = requests.post(
    f"{host}/api/2.0/apps/{APP_NAME}/deployments",
    headers=H,
    json={"source_code_path": src_fs, "mode": "SNAPSHOT"},
)
assert r.status_code in (200, 201), f"Deploy failed: {r.status_code} {r.text}"
deployment_id = r.json().get("deployment_id", "")
print(f"       Deployment started: {deployment_id}")

last_state = ""
for _ in range(60):
    time.sleep(10)
    info = requests.get(f"{host}/api/2.0/apps/{APP_NAME}", headers=H).json()
    dep = info.get("active_deployment") or info.get("pending_deployment") or {}
    status = dep.get("status", {})
    state = status.get("state", "")
    msg = status.get("message", "")
    if state != last_state:
        print(f"       {state}: {msg}")
        last_state = state
    if state == "SUCCEEDED":
        url = info.get("url", "")
        app_state = info.get("app_status", {}).get("state", "")
        print(f"[4/4] Deployed (app_status={app_state}).")
        print(f"\n{'='*60}\n  App URL: {url}\n{'='*60}")
        try:
            displayHTML(
                f'<h2 style="color:#4ECDC4">Model Viewer Deployed!</h2>'
                f'<p style="font-size:16px"><a href="{url}" target="_blank">{url}</a></p>'
                f'<p>Upload a <code>model.json</code> to visualize your data model.</p>'
            )
        except Exception:
            pass
        break
    if state == "FAILED":
        raise Exception(f"Deployment failed: {msg}")
else:
    raise Exception("Deployment did not complete within 10 minutes. Check app status in the workspace UI.")